# Лабораторная работа №2
## Прогноз энергопотребления интеллектуального учебного пространства

**Курс:** Технологии машинного обучения
**Направление:** 44.04.01 Педагогическое образование — «Умные системы и интернет вещей в образовании»
**Раздел 2.** Машинное обучение · **20 баллов БРС**

---

### Сценарий

Школьная робототехническая лаборатория совмещена с малой серверной: 3D-принтеры, паяльные станции, зарядные стойки для конструкторов, локальный сервер и NAS. Профиль потребления рваный — базовая нагрузка 40–70 Вт·ч за 10-минутный интервал, но при запуске печати или одновременной зарядке всех наборов возникают всплески до 1000+ Вт·ч.

Управляющая система должна **за 10 минут до факта** знать ожидаемую нагрузку, чтобы:

* заранее включить приточную вентиляцию серверной (тепло идёт за нагрузкой с задержкой);
* сгладить пики, отложив некритичные задачи на ночной тариф;
* предупредить лаборанта о риске превышения лимита вводного автомата;
* вести энергетический паспорт помещения.

### Формальная постановка

$$\hat{y}_{t+h} = f_\theta\big(y_t, y_{t-1}, \dots, y_{t-p},\ \mathbf{s}_t,\ \mu_w(t), \sigma_w(t),\ c_t\big), \qquad h = 1 \;(10\text{ мин}).$$

**Критическое ограничение:** каждый признак в момент $t$ обязан быть вычислим только по данным, поступившим **до момента $t$ включительно**.

### Что нужно сделать

| Блок | Действие | Балл |
|---|---|---|
| 1 | Временна́я ось: разбор `date`, сортировка, проверка регулярности шага, пропуски | 3 |
| 2 | Лаговые и оконные признаки, календарь | 4 |
| 3 | Наивный baseline + три модели варианта | 4 |
| 4 | `TimeSeriesSplit(n_splits=5)`, метрики по фолдам | 3 |
| 5 | MAE / RMSE / $R^2$, сравнение, график факт-прогноз | 3 |
| 6 | Важности признаков, выводы для энергосбережения | 2 |
| 7 | Воспроизводимость | 1 |

**Шесть блоков `# TODO: СТУДЕНТ` и шесть тестов `TEST`. `TEST 2` — автоматический детектор look-ahead bias.**

## Блок 0. Теория

### 0.1. Чем временной ряд IoT отличается от обычной таблицы

В ЛР №1 строки выборки считались независимыми — это допущение i.i.d. Для телеметрии оно **неверно**, и почти все привычные приёмы ломаются:

| Свойство | Обычная таблица | Временной ряд IoT |
|---|---|---|
| Порядок строк | Не значим | **Несущий смысл** |
| Независимость наблюдений | Предполагается | Нарушена (автокорреляция) |
| Перемешивание при сплите | Норма | **Запрещено** |
| Источник информации | Только текущая строка | Строка + вся история до неё |
| Пропуск | Отсутствующее значение | **Разрыв в сетке времени** |

Автокорреляция означает, что $y_t$ и $y_{t-1}$ несут почти одинаковую информацию. Отсюда два следствия: (а) наивный прогноз «завтра как вчера» неожиданно силён и обязан использоваться как baseline; (б) случайный сплит превращает прогнозирование в интерполяцию и завышает метрики.

### 0.2. Look-ahead bias

**Look-ahead bias** — использование информации, которой в момент прогноза ещё не существовало. Внешне выглядит как отличное качество модели, на объекте даёт полный отказ.

> **Правило допустимого признака.** Признак $x_t$ допустим тогда и только тогда, когда его можно вычислить, имея лишь поток данных, поступивший к моменту $t$.

Четыре типовых источника:

**1. Перемешивание выборки.** `train_test_split(shuffle=True)`, `KFold`, `cross_val_score(cv=5)` — все они перемешивают. Обучение на мае, тест на январе — это не прогноз.

**2. Центрированное окно.**
```python
df['roll'] = df['Appliances'].rolling(12, center=True).mean()   # ЗАПРЕЩЕНО
```
`center=True` включает в среднее 6 будущих отсчётов. Код выглядит нормально — тем и опасен.

**3. Интерполяция по всему ряду.** `interpolate()` протягивает будущее в прошлое; `fillna(df.mean())` использует среднее, посчитанное в том числе по тесту. В реальном времени реализуем только `ffill` — протяжка последнего известного значения.

**4. Масштабирование до сплита.** `StandardScaler().fit_transform(X)` на всём ряде переносит статистику теста в обучение. Лечение — `Pipeline`, где `fit` вызывается на каждом обучающем фолде отдельно.

### 0.3. Генерация признаков

При корректной постановке модель видит только «хвост» истории, поэтому его нужно явно свернуть в признаки.

**Лаги** — сдвиг ряда назад:
$$x^{(k)}_t = y_{t-k}, \qquad k = 0, 1, \dots, p.$$
Лаг нулевого порядка $y_t$ доступен (мы прогнозируем $y_{t+1}$) и представляет собой в точности наивный прогноз.

**Оконные статистики** — свёртка последних $w$ отсчётов:
$$\mu_w(t) = \frac{1}{w}\sum_{j=0}^{w-1} y_{t-j}, \qquad
\sigma_w(t) = \sqrt{\frac{1}{w-1}\sum_{j=0}^{w-1}\big(y_{t-j} - \mu_w(t)\big)^2}.$$
Среднее описывает уровень нагрузки, стандартное отклонение — «нервозность» режима (идёт печать или помещение простаивает), максимум — недавний пик, разность $y_t - y_{t-w}$ — тренд.

**Календарные признаки.** Час суток и день недели задают расписание занятий. Час — величина **циклическая**: 23:50 и 00:10 разделены 20 минутами, а не 23 часами. Числовое кодирование этого не отражает, поэтому применяется тригонометрическое:
$$\text{hour\_sin} = \sin\frac{2\pi \cdot \text{hour}}{24}, \qquad
\text{hour\_cos} = \cos\frac{2\pi \cdot \text{hour}}{24}.$$
Пара $(\sin, \cos)$ помещает часы на окружность, и расстояние между 23:50 и 00:10 становится малым.

### 0.4. Валидация: `TimeSeriesSplit(n_splits=5)`

Единственная разрешённая схема — расширяющееся окно:

```
Фолд 1: [train    ][test]
Фолд 2: [train         ][test]
Фолд 3: [train              ][test]
Фолд 4: [train                   ][test]
Фолд 5: [train                        ][test]
                                              время →
```

Каждый фолд — самостоятельный эксперимент «обучились на накопленной истории, спрогнозировали следующий период». Пять фолдов дают не одно число, а **распределение** метрики. Разброс между фолдами информативен сам по себе: если качество деградирует к последнему фолду, модель нестабильна во времени и в эксплуатацию не годится.

### 0.5. Метрики

$$\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}\big|y_i - \hat{y}_i\big|, \qquad
\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\big(y_i - \hat{y}_i\big)^2}, \qquad
R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}.$$

* **MAE** (Вт·ч) — средняя ошибка, устойчива к выбросам, отвечает вопросу учёта расхода.
* **RMSE** (Вт·ч) — квадратичный штраф, чувствительна к промахам по пикам, отвечает вопросу защиты вводного автомата.
* **$R^2$** — доля объяснённой дисперсии; **может быть отрицательным**, если модель хуже константного прогноза средним.

Ранжирование моделей по MAE и по RMSE может **не совпадать** — обе метрики обязаны быть в отчёте. И главное: на автокоррелированном ряде высокий $R^2$ сам по себе ничего не значит, единственная честная точка отсчёта — наивный прогноз $\hat{y}_{t+1} = y_t$.

## Блок 1. Окружение, вариант, данные

In [ ]:
!pip install -q ucimlrepo scikit-learn pandas matplotlib

In [ ]:
import sys, io, zipfile, urllib.request, warnings, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

warnings.filterwarnings("ignore")

# --- ВОСПРОИЗВОДИМОСТЬ (критерий 7) ---
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)

print("Python      :", sys.version.split()[0], "|", platform.system())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("SEED        :", SEED)

In [ ]:
# ======================================================================
#  УКАЖИТЕ СВОЙ НОМЕР ВАРИАНТА ИЗ ВЕДОМОСТИ (1..25)
# ======================================================================
VARIANT = 1   # <-- ИЗМЕНИТЕ НА СВОЙ НОМЕР
# ======================================================================

In [ ]:
# Справочники параметризации (раздел 6.1 методички Readme_lab_02.md).
WINDOWS = [3, 6, 12, 24]          # отсчёты по 10 минут: 30 мин, 1 ч, 2 ч, 4 ч

ZONE_SETS = {
    "R1": ["T1", "RH_1", "T2", "RH_2", "T_out", "RH_out"],
    "R2": ["T3", "RH_3", "T4", "RH_4", "T_out", "Windspeed"],
    "R3": ["T5", "RH_5", "T6", "RH_6", "T_out", "Press_mm_hg"],
    "R4": ["T7", "RH_7", "T8", "RH_8", "T_out", "Tdewpoint"],
    "R5": ["T3", "RH_3", "T9", "RH_9", "T_out", "RH_out"],
}
ZONE_ORDER = ["R1", "R2", "R3", "R4", "R5"]

TARGET = "Appliances"
HORIZON = 1          # прогноз на 1 шаг = 10 минут вперёд (одинаково для всех вариантов)
CONTROL_NOISE = "rv1"   # контрольный шумовой признак датасета


def get_variant_config(v: int) -> dict:
    """Конфигурация варианта по правилам раздела 6.1.

    lag    = 1 + (v mod 6)
    window = WINDOWS[(v + 1) mod 4]
    зоны   = ZONE_SETS[ZONE_ORDER[(v - 1) mod 5]]
    линейная модель: Ridge для нечётных v, Lasso для чётных
    бустинг: HistGradientBoosting, кроме v кратных 3 -> GradientBoosting
    """
    assert isinstance(v, int) and 1 <= v <= 25, "VARIANT должен быть целым числом 1..25"
    zid = ZONE_ORDER[(v - 1) % 5]
    return {
        "variant": v,
        "lag": 1 + (v % 6),
        "window": WINDOWS[(v + 1) % 4],
        "zone_set_id": zid,
        "sensors": ZONE_SETS[zid],
        "linear": "Ridge" if v % 2 == 1 else "Lasso",
        "bagging": "RandomForestRegressor",
        "boosting": "HistGradientBoostingRegressor" if v % 3 != 0 else "GradientBoostingRegressor",
        "horizon": HORIZON,
        "random_state": SEED + v,
    }


CFG = get_variant_config(VARIANT)

print("=" * 66)
print(f"  КОНФИГУРАЦИЯ ВАРИАНТА №{CFG['variant']}")
print("=" * 66)
print(f"  Глубина лага   : {CFG['lag']}   [= 1 + ({VARIANT} mod 6 = {VARIANT % 6})]")
print(f"  Окно           : {CFG['window']} отсчётов = {CFG['window'] * 10} минут")
print(f"  Зоны           : {CFG['zone_set_id']} -> {CFG['sensors']}")
print(f"  Линейная модель: {CFG['linear']}")
print(f"  Бэггинг        : {CFG['bagging']}")
print(f"  Бустинг        : {CFG['boosting']}")
print(f"  Горизонт       : {CFG['horizon']} шаг = {CFG['horizon'] * 10} минут")
print("=" * 66)

### 1.1. Загрузка данных

Три пути с автопереключением: `ucimlrepo(id=235)` → прямая ссылка UCI → синтетический генератор-фолбэк. Фолбэк воспроизводит статистику реального ряда (медиана ≈ 70 Вт·ч, среднее ≈ 97 Вт·ч, максимум 1080 Вт·ч, сильная правая асимметрия) и нужен только для отладки при отсутствии сети.

> Если сработал фолбэк — **обязательно укажите это в выводах**. Фактический источник хранится в `DATA_SOURCE`.

In [ ]:
UCI_ZIP = "https://archive.ics.uci.edu/static/public/235/appliances+energy+prediction.zip"


def make_synthetic_appliances(n=19735, seed=SEED):
    """Физически мотивированный генератор-фолбэк (профиль здания + метео)."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2016-01-11 17:00:00", periods=n, freq="10min")
    minute = idx.hour * 60 + idx.minute
    doy = idx.dayofyear.values
    wd = idx.dayofweek.values

    season = 6.0 * np.sin(2 * np.pi * (doy - 100) / 365.0)
    day_out = 4.5 * np.sin(2 * np.pi * (minute - 480) / 1440.0)
    T_out = 4.0 + season + day_out + rng.normal(0, 1.1, n)
    RH_out = np.clip(80 - 1.6 * (T_out - 4) + rng.normal(0, 7, n), 25, 100)
    Windspeed = np.clip(rng.gamma(2.0, 1.8, n), 0, 14)
    Press = 755 + 6 * np.sin(2 * np.pi * doy / 60.0) + rng.normal(0, 2.5, n)
    Visibility = np.clip(40 - 0.25 * RH_out + rng.normal(0, 6, n), 1, 66)
    Tdew = T_out - (100 - RH_out) / 5.0

    work = ((minute >= 8 * 60) & (minute <= 21 * 60) & (wd < 6)).astype(float)
    evening = np.exp(-((minute - 19 * 60) ** 2) / (2 * 110 ** 2))
    morning = np.exp(-((minute - 8 * 60) ** 2) / (2 * 70 ** 2))
    activity = 0.15 + 0.55 * work * (0.4 + 0.9 * evening + 0.7 * morning)

    occ, st = np.zeros(n), 0.0
    for i in range(n):
        st = 0.82 * st + 0.18 * (rng.random() < activity[i])
        occ[i] = st

    base = 30.0
    raw_spikes = (rng.random(n) < 0.045 * (0.15 + activity)) * rng.gamma(1.7, 165, n)
    ker = np.array([1.0, 0.85, 0.62, 0.40, 0.22, 0.10])       # прибор работает несколько интервалов
    spikes = np.convolve(raw_spikes, ker, mode="full")[:n]
    app = (base + 58 * occ + 6.5 * np.maximum(0, 12 - T_out) * occ
           + spikes + rng.gamma(1.6, 5, n))
    Appliances = np.clip(np.round(app / 10) * 10, 10, 1080)
    lights = np.clip(np.round((occ * 40 * (1 - np.clip((T_out + 8) / 25, 0, 1))
                               + rng.normal(0, 4, n)) / 10) * 10, 0, 70)

    out = {"date": idx, "Appliances": Appliances, "lights": lights}
    room_base = {1: 21.6, 2: 20.3, 3: 22.3, 4: 20.8, 5: 19.6, 6: 7.9, 7: 20.3, 8: 22.0, 9: 19.5}
    for k, b in room_base.items():
        inertia = 0.35 if k != 6 else 0.9      # зона 6 — наружный датчик
        t = (b + (1 - inertia) * (1.4 * occ + 0.35 * day_out)
             + inertia * 0.55 * (T_out - 4) + rng.normal(0, 0.22, n))
        rh = np.clip(38 + 4.5 * occ - 0.9 * (t - b) + 0.18 * (RH_out - 70)
                     + rng.normal(0, 1.6, n), 15, 65)
        out[f"T{k}"] = np.round(t, 3)
        out[f"RH_{k}"] = np.round(rh, 3)
    out.update({"T_out": np.round(T_out, 3), "Press_mm_hg": np.round(Press, 2),
                "RH_out": np.round(RH_out, 1), "Windspeed": np.round(Windspeed, 3),
                "Visibility": np.round(Visibility, 2), "Tdewpoint": np.round(Tdew, 3)})
    df = pd.DataFrame(out)
    df["rv1"] = rng.random(n) * 50      # контрольный шум (как в оригинальном датасете)
    df["rv2"] = df["rv1"]
    return df


def load_energy(verbose=True):
    """Возвращает (DataFrame, source): 'ucimlrepo' | 'uci_zip' | 'synthetic'."""
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=235)
        df = pd.concat([ds.data.features.copy(), ds.data.targets.copy()], axis=1)
        df.columns = [c.strip() for c in df.columns]
        if verbose:
            print("[OK] Источник: ucimlrepo (fetch_ucirepo(id=235))")
        return df, "ucimlrepo"
    except Exception as e:
        if verbose:
            print(f"[!] ucimlrepo недоступен: {type(e).__name__}: {e}")

    try:
        with urllib.request.urlopen(UCI_ZIP, timeout=30) as r:
            blob = r.read()
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            name = [n for n in z.namelist() if n.endswith(".csv")][0]
            with z.open(name) as fh:
                df = pd.read_csv(fh)
        df.columns = [c.strip().strip('"') for c in df.columns]
        if verbose:
            print("[OK] Источник: прямой ZIP archive.ics.uci.edu")
        return df, "uci_zip"
    except Exception as e:
        if verbose:
            print(f"[!] Прямая загрузка недоступна: {type(e).__name__}: {e}")

    if verbose:
        print("[i] Включён СИНТЕТИЧЕСКИЙ ФОЛБЭК — отразите это в выводах!")
    return make_synthetic_appliances(), "synthetic"


df_raw, DATA_SOURCE = load_energy()
print(f"\nЗагружено: {df_raw.shape[0]} строк, {df_raw.shape[1]} столбцов")
print("Столбцы:", list(df_raw.columns))
df_raw.head(3)

## Блок 2. Временна́я ось — 3 балла

Первое, что делают с телеметрией: приводят время в порядок. Пока не доказано, что ряд отсортирован и сетка регулярна, любые лаги и окна считать нельзя — `shift(1)` на неотсортированном ряде даст бессмысленный признак.

In [ ]:
# 2.1. Разбор даты, сортировка, проверка регулярности сетки.
df = df_raw.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

deltas = df["date"].diff().dropna()
step = deltas.mode()[0]

print(f"Период наблюдений : {df['date'].min()} — {df['date'].max()}")
print(f"Наблюдений        : {len(df)}")
print(f"Модальный шаг     : {step}")
print(f"Уникальных шагов  : {deltas.nunique()}")
print("\nРаспределение интервалов между соседними отсчётами:")
print(deltas.value_counts().head(5).to_string())

expected = int((df["date"].max() - df["date"].min()) / step) + 1
gaps = deltas[deltas > step]
print(f"\nОжидалось отсчётов при идеальной сетке : {expected}")
print(f"Фактически                             : {len(df)}")
print(f"Разрывов (интервал > {step})            : {len(gaps)}")
if len(gaps):
    print(f"Максимальный разрыв: {gaps.max()}")
print(f"\nДубликатов по времени: {int(df['date'].duplicated().sum())}")
print(f"Пропущенных значений в таблице: {int(df.isna().sum().sum())}")

**ВЫВОД 2.1 (заполните):**
_Регулярна ли сетка? Сколько разрывов и какой максимальный? Что произойдёт с признаком `shift(6)`, если в ряде есть разрыв в 3 часа? Почему для восстановления сетки допустим только `ffill`, но не `interpolate()`?_

In [ ]:
# 2.2. Приведение к регулярной сетке.
# ВНИМАНИЕ: reindex + ffill — единственный способ, реализуемый в реальном времени.
# interpolate() протянул бы будущее в прошлое (look-ahead bias, источник 3).
full_index = pd.date_range(df["date"].min(), df["date"].max(), freq=step)
df = (df.set_index("date")
        .reindex(full_index)
        .ffill()
        .rename_axis("date")
        .reset_index())

print(f"После восстановления сетки: {len(df)} отсчётов, "
      f"пропусков {int(df.isna().sum().sum())}")
print(f"Шаг регулярен: {df['date'].diff().dropna().nunique() == 1}")

In [ ]:
# 2.3. Целевой ряд: общий вид, распределение, суточный и недельный профиль.
fig, ax = plt.subplots(2, 1, figsize=(12, 6.5), sharex=False)
ax[0].plot(df["date"], df[TARGET], lw=0.4, color="#C44E52")
ax[0].set_title(f"Энергопотребление {TARGET}, весь период")
ax[0].set_xlabel("Время"); ax[0].set_ylabel("Вт·ч за 10 мин")

win = df.iloc[3000:3000 + 6 * 24 * 7]      # одна неделя крупным планом
ax[1].plot(win["date"], win[TARGET], lw=0.8, color="#4C72B0")
ax[1].set_title("Фрагмент: одна неделя")
ax[1].set_xlabel("Время"); ax[1].set_ylabel("Вт·ч за 10 мин")
ax[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

print(df[TARGET].describe().round(1).to_string())
print(f"\nАсимметрия (skew): {df[TARGET].skew():.2f}  "
      f"(> 1 означает сильный правый хвост — редкие мощные пики)")
print(f"Доля интервалов с потреблением выше 200 Вт·ч: "
      f"{(df[TARGET] > 200).mean() * 100:.2f} %")

In [ ]:
# 2.4. Автокорреляция цели — численное обоснование запрета на перемешивание.
print("Автокорреляция Appliances:")
for lag in (1, 3, 6, 18, 36, 144):
    print(f"  лаг {lag:3d} отсчётов ({lag * 10:4d} мин): {df[TARGET].autocorr(lag):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
lags = range(1, 145)
ax[0].plot(list(lags), [df[TARGET].autocorr(l) for l in lags], lw=1.4, color="#55A868")
ax[0].axhline(0, color="k", lw=1)
ax[0].set_title("Автокорреляционная функция Appliances")
ax[0].set_xlabel("Лаг, отсчёты по 10 мин"); ax[0].set_ylabel("Коэффициент автокорреляции")

hour_profile = df.groupby(df["date"].dt.hour)[TARGET].mean()
wd_profile = df.groupby(df["date"].dt.dayofweek)[TARGET].mean()
ax[1].plot(hour_profile.index, hour_profile.values, marker="o", lw=1.6,
           color="#C44E52", label="по часам суток")
ax[1].set_title("Средняя нагрузка по часам суток")
ax[1].set_xlabel("Час"); ax[1].set_ylabel("Вт·ч за 10 мин"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("\nСредняя нагрузка по дням недели (0 = понедельник):")
print(wd_profile.round(1).to_string())

**ВЫВОД 2.2 (заполните):**
_Чему равна автокорреляция на лаге 1? Что это означает для наивного прогноза? Виден ли суточный цикл на ACF (пик около лага 144)? Как суточный и недельный профили соотносятся с расписанием занятий?_

## Блок 3. Генерация признаков — 4 балла

Модель не видит истории — её нужно свернуть в признаки явно. Ниже три `TODO`: лаги, оконные статистики, календарь. **Главное требование: ни один признак не должен заглядывать в будущее.** Проверка автоматизирована в `TEST 2`.

In [ ]:
# TODO-1: СТУДЕНТ — ЛАГОВЫЕ ПРИЗНАКИ
#
# Реализуйте add_lag_features(F, df, col, lag):
#   добавляет в F столбцы f'{col}_lag0', f'{col}_lag1', ..., f'{col}_lag{lag}',
#   где f'{col}_lag{k}' = df[col].shift(k).
# Напоминание: lag0 = текущее значение (оно доступно, т.к. прогнозируем t+1)
#              и одновременно является наивным прогнозом.

def add_lag_features(F: pd.DataFrame, df: pd.DataFrame, col: str, lag: int) -> pd.DataFrame:
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError("Реализуйте add_lag_features")

In [ ]:
# TODO-2: СТУДЕНТ — ОКОННЫЕ СТАТИСТИКИ
#
# Реализуйте add_window_features(F, df, col, window):
#   f'{col}_rmean{window}'  — скользящее среднее по окну
#   f'{col}_rstd{window}'   — скользящее стандартное отклонение
#   f'{col}_rmax{window}'   — скользящий максимум
#   f'{col}_delta{window}'  — разность df[col] - df[col].shift(window) (тренд)
#
# ЗАПРЕЩЕНО: rolling(window, center=True) — это look-ahead bias.
# Окно должно заканчиваться в текущей точке t (поведение rolling по умолчанию).

def add_window_features(F: pd.DataFrame, df: pd.DataFrame, col: str, window: int) -> pd.DataFrame:
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError("Реализуйте add_window_features")

In [ ]:
# TODO-3: СТУДЕНТ — КАЛЕНДАРНЫЕ ПРИЗНАКИ
#
# Реализуйте add_calendar_features(F, df):
#   'hour'       — час с дробной частью: dt.hour + dt.minute / 60
#   'dayofweek'  — день недели 0..6
#   'is_weekend' — 1 для субботы и воскресенья, иначе 0
#   'hour_sin'   — sin(2*pi*hour/24)      <- циклическое кодирование
#   'hour_cos'   — cos(2*pi*hour/24)
# Зачем sin/cos: 23:50 и 00:10 разделены 20 минутами, а числа 23.83 и 0.17 — почти сутками.

def add_calendar_features(F: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError("Реализуйте add_calendar_features")

In [ ]:
# 3.1. Сборка матрицы признаков (готовый код, использует ваши три функции).
def build_feature_frame(df: pd.DataFrame, sensors, lag: int, window: int) -> pd.DataFrame:
    """Только признаки, без целевой переменной. Все столбцы — по данным до t включительно."""
    F = pd.DataFrame(index=df.index)
    F = add_lag_features(F, df, TARGET, lag)
    F = add_window_features(F, df, TARGET, window)
    for c in sensors:
        F[c] = df[c].values                        # текущее показание доступно в момент t
        F[f"{c}_lag{lag}"] = df[c].shift(lag).values
        F[f"{c}_rmean{window}"] = df[c].rolling(window).mean().values
        F[f"{c}_delta{window}"] = (df[c] - df[c].shift(window)).values
    F[CONTROL_NOISE] = df[CONTROL_NOISE].values    # контрольный шум для отбора признаков
    F = add_calendar_features(F, df)
    return F


def build_dataset(df: pd.DataFrame, sensors, lag: int, window: int, horizon: int):
    """Возвращает (X, y, idx): признаки, цель y_{t+horizon} и соответствующие метки времени."""
    F = build_feature_frame(df, sensors, lag, window)
    y = df[TARGET].shift(-horizon).rename("target")
    data = pd.concat([F, y, df["date"].rename("ts")], axis=1).dropna()
    return data.drop(columns=["target", "ts"]), data["target"], data["ts"]


X, y, ts = build_dataset(df, CFG["sensors"], CFG["lag"], CFG["window"], CFG["horizon"])

print(f"Матрица признаков: {X.shape[0]} строк, {X.shape[1]} признаков")
print(f"Потеряно строк при dropna: {len(df) - len(X)} "
      f"(разгон окна {CFG['window']} + горизонт {CFG['horizon']})")
print(f"Период: {ts.min()} — {ts.max()}")
print("\nСписок признаков:")
for i, c in enumerate(X.columns, 1):
    print(f"  {i:2d}. {c}")

In [ ]:
# ============ TEST 1 — структура матрицы признаков ============
def test_1_features():
    lag, w = CFG["lag"], CFG["window"]
    for k in range(lag + 1):
        assert f"{TARGET}_lag{k}" in X.columns, f"Нет лага {TARGET}_lag{k}"
    for suf in (f"_rmean{w}", f"_rstd{w}", f"_rmax{w}", f"_delta{w}"):
        assert TARGET + suf in X.columns, f"Нет оконного признака {TARGET}{suf}"
    for c in ("hour", "dayofweek", "is_weekend", "hour_sin", "hour_cos"):
        assert c in X.columns, f"Нет календарного признака {c}"
    for s in CFG["sensors"]:
        assert s in X.columns, f"Нет сенсора {s} из набора {CFG['zone_set_id']}"
    assert CONTROL_NOISE in X.columns, "Контрольный шум rv1 должен быть в признаках"
    assert TARGET not in X.columns, "Сама цель Appliances не может быть признаком без сдвига"
    assert X.isna().sum().sum() == 0, "В матрице признаков остались NaN"
    assert np.allclose(X["hour_sin"] ** 2 + X["hour_cos"] ** 2, 1.0), \
        "sin^2 + cos^2 должно равняться 1 — проверьте циклическое кодирование"
    assert len(X) == len(y) == len(ts)
    print(f"✅ TEST 1 PASSED — {X.shape[1]} признаков, структура корректна")

test_1_features()

In [ ]:
# ============ TEST 2 — ДЕТЕКТОР LOOK-AHEAD BIAS ============
# Идея: признаки в точке t, посчитанные по полному ряду и по ряду, обрезанному
# на этой же точке, обязаны совпадать. Если не совпали — признак видит будущее.
def test_2_no_lookahead():
    probes = [500, 5000, 12000, len(df) - 50]
    F_full = build_feature_frame(df, CFG["sensors"], CFG["lag"], CFG["window"])
    bad = []
    for t in probes:
        df_trunc = df.iloc[:t + 1].copy().reset_index(drop=True)
        F_tr = build_feature_frame(df_trunc, CFG["sensors"], CFG["lag"], CFG["window"])
        a, b = F_full.iloc[t], F_tr.iloc[t]
        diff = (a - b).abs()
        bad += [c for c in F_full.columns if not (np.isnan(a[c]) and np.isnan(b[c]))
                and not (diff[c] < 1e-9)]
    bad = sorted(set(bad))
    assert not bad, ("LOOK-AHEAD BIAS в признаках: " + ", ".join(bad) +
                     "\nВероятная причина: rolling(..., center=True) или shift с отрицательным сдвигом")
    print(f"✅ TEST 2 PASSED — все {F_full.shape[1]} признаков вычислимы "
          f"только по прошлому (проверено в {len(probes)} точках)")

test_2_no_lookahead()

## Блок 4. Baseline и модели — 4 балла

In [ ]:
# TODO-4: СТУДЕНТ — НАИВНЫЙ BASELINE
#
# Наивный прогноз: y_hat(t+1) = y(t), то есть просто столбец f'{TARGET}_lag0'.
# Реализуйте naive_predict(X_part) -> np.ndarray с прогнозом для переданного среза X.
# Это одна строка кода, но без неё вся работа теряет смысл: только baseline
# показывает, сколько РЕАЛЬНО добавляет машинное обучение.

def naive_predict(X_part: pd.DataFrame) -> np.ndarray:
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError("Реализуйте naive_predict")

In [ ]:
# ============ TEST 3 — наивный прогноз ============
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def test_3_naive():
    p = naive_predict(X)
    assert isinstance(p, np.ndarray) and p.shape == (len(X),), "Ожидался np.ndarray длины len(X)"
    assert np.allclose(p, X[f"{TARGET}_lag0"].values), "Наивный прогноз = текущее значение ряда"
    mae = mean_absolute_error(y, p)
    assert mae > 0, "MAE наивного прогноза не может быть нулевым"
    print(f"✅ TEST 3 PASSED — наивный прогноз: MAE = {mae:.2f} Вт·ч, "
          f"R2 = {r2_score(y, p):.4f}")
    print("   Это ПЛАНКА: модель, не обошедшая её, не нужна.")

test_3_naive()

In [ ]:
# 4.1. Фабрика моделей варианта.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import (RandomForestRegressor, HistGradientBoostingRegressor,
                              GradientBoostingRegressor)
from sklearn.model_selection import TimeSeriesSplit


def build_model(name: str, random_state: int = SEED) -> Pipeline:
    """Модель, обёрнутая в Pipeline.

    Масштабирование включено ТОЛЬКО для линейных моделей: их регуляризация
    зависит от масштаба признаков. Деревьям оно не нужно и лишь замедляет обучение.
    Pipeline обязателен, потому что fit скейлера должен выполняться
    на каждом обучающем фолде отдельно (иначе look-ahead bias, источник 4).
    """
    if name == "Ridge":
        return Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
    if name == "Lasso":
        return Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.1, max_iter=5000))])
    if name == "RandomForestRegressor":
        return Pipeline([("model", RandomForestRegressor(
            n_estimators=150, min_samples_leaf=2, max_features=0.5,
            n_jobs=-1, random_state=random_state))])
    if name == "HistGradientBoostingRegressor":
        return Pipeline([("model", HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.06, random_state=random_state))])
    if name == "GradientBoostingRegressor":
        return Pipeline([("model", GradientBoostingRegressor(
            n_estimators=250, learning_rate=0.06, max_depth=3, random_state=random_state))])
    raise ValueError(f"Неизвестная модель: {name}")


MODEL_NAMES = [CFG["linear"], CFG["bagging"], CFG["boosting"]]
print("Модели варианта:", MODEL_NAMES)
for m in MODEL_NAMES:
    print(" ", build_model(m))

### 4.2. Кросс-валидация `TimeSeriesSplit`

```
Фолд 1: [train    ][test]
Фолд 2: [train         ][test]
Фолд 3: [train              ][test]
Фолд 4: [train                   ][test]
Фолд 5: [train                        ][test]
                                              время →
```

`TimeSeriesSplit` не перемешивает данные никогда — само его применение закрывает первый источник look-ahead bias. Обратите внимание: первый фолд обучается на существенно меньшем объёме, поэтому метрики на нём обычно хуже. Это нормально и должно быть прокомментировано, а не «исправлено».

In [ ]:
# TODO-5: СТУДЕНТ — ЦИКЛ КРОСС-ВАЛИДАЦИИ
#
# Реализуйте cross_validate_ts(name, model_or_none, X, y, tscv):
#   для каждого фолда (номер с 1):
#     * если model_or_none is None -> прогноз naive_predict(X.iloc[test])
#       иначе: клонировать модель (sklearn.base.clone), обучить на train, предсказать test
#     * посчитать MAE, RMSE = sqrt(MSE), R2
#     * добавить в список словарь:
#       {'Модель': name, 'Фолд': k, 'n_train': ..., 'n_test': ...,
#        'MAE': ..., 'RMSE': ..., 'R2': ...}
#   вернуть список словарей.
#
# ВАЖНО: обязательно clone(model) на каждом фолде, иначе модель дообучается поверх
# предыдущего фолда и результаты будут неверными.

from sklearn.base import clone


def cross_validate_ts(name, model_or_none, X, y, tscv) -> list:
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError("Реализуйте cross_validate_ts")


tscv = TimeSeriesSplit(n_splits=5)
records = []
records += cross_validate_ts("Naive (y_t)", None, X, y, tscv)
for name in MODEL_NAMES:
    print(f"Обучение {name} ...")
    records += cross_validate_ts(name, build_model(name, CFG["random_state"]), X, y, tscv)

folds_df = pd.DataFrame(records)
print("\nМЕТРИКИ ПО КАЖДОМУ ФОЛДУ:")
folds_df.round(3)

In [ ]:
# ============ TEST 4 — корректность кросс-валидации ============
def test_4_cv():
    assert len(folds_df) == 4 * 5, "Ожидалось 4 модели (включая naive) x 5 фолдов = 20 строк"
    assert folds_df["Фолд"].nunique() == 5, "Должно быть ровно 5 фолдов"
    g = folds_df[folds_df["Модель"] == "Naive (y_t)"].sort_values("Фолд")
    assert g["n_train"].is_monotonic_increasing, \
        "Размер train должен РАСТИ от фолда к фолду (расширяющееся окно TimeSeriesSplit)"
    assert g["n_test"].nunique() == 1, "Размер test у TimeSeriesSplit одинаков на всех фолдах"
    assert (folds_df["RMSE"] >= folds_df["MAE"] - 1e-9).all(), \
        "RMSE не может быть меньше MAE — проверьте формулы"
    assert folds_df["R2"].max() < 0.98, \
        "R2 близок к 1 — почти наверняка look-ahead bias, перепроверьте признаки"
    print("✅ TEST 4 PASSED — 5 фолдов, окно расширяется, метрики согласованы")

test_4_cv()

## Блок 5. Сравнение моделей — 3 балла

In [ ]:
# 5.1. Сводная таблица: среднее по фолдам и разброс.
summary = (folds_df.groupby("Модель")[["MAE", "RMSE", "R2"]]
           .agg(["mean", "std"]).round(3))
summary.columns = [f"{a}_{b}" for a, b in summary.columns]
summary = summary.sort_values("RMSE_mean")

naive_mae = summary.loc["Naive (y_t)", "MAE_mean"]
naive_rmse = summary.loc["Naive (y_t)", "RMSE_mean"]
summary["Прирост MAE, %"] = ((naive_mae - summary["MAE_mean"]) / naive_mae * 100).round(2)
summary["Прирост RMSE, %"] = ((naive_rmse - summary["RMSE_mean"]) / naive_rmse * 100).round(2)

print("СВОДКА (среднее ± std по 5 фолдам), отсортировано по RMSE:")
summary

In [ ]:
# 5.2. Визуальное сравнение: метрики по фолдам.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    for name, g in folds_df.groupby("Модель"):
        style = dict(marker="s", lw=2.2, ls="--") if name.startswith("Naive") else dict(marker="o", lw=1.6)
        ax.plot(g["Фолд"], g[metric], label=name, **style)
    ax.set_title(f"{metric} по фолдам")
    ax.set_xlabel("Номер фолда TimeSeriesSplit"); ax.set_ylabel(metric)
    ax.set_xticks(sorted(folds_df["Фолд"].unique()))
axes[0].legend(fontsize=7.5, loc="best")
plt.tight_layout(); plt.show()

In [ ]:
# ============ TEST 5 — сравнение с baseline ============
def test_5_comparison():
    best = summary.drop(index="Naive (y_t)")["RMSE_mean"].min()
    assert best < naive_rmse, \
        ("Ни одна модель не обошла наивный прогноз по RMSE. "
         "Это возможно, но требует явного объяснения в выводах — проверьте признаки и параметры.")
    assert (summary["R2_mean"] < 0.98).all(), "R2 подозрительно высок — ищите утечку"
    assert summary["MAE_std"].max() < summary["MAE_mean"].max(), \
        "Разброс MAE между фолдами сопоставим со средним — модель нестабильна"
    print(f"✅ TEST 5 PASSED — лучший RMSE = {best:.2f} против наивного {naive_rmse:.2f} Вт·ч "
          f"(выигрыш {(naive_rmse - best) / naive_rmse * 100:.1f} %)")

test_5_comparison()

In [ ]:
# 5.3. Факт против прогноза на последнем (самом «свежем») фолде.
splits = list(tscv.split(X))
tr_idx, te_idx = splits[-1]
best_name = summary.drop(index="Naive (y_t)").index[0]
best_model = build_model(best_name, CFG["random_state"]).fit(X.iloc[tr_idx], y.iloc[tr_idx])
pred = best_model.predict(X.iloc[te_idx])
pred_naive = naive_predict(X.iloc[te_idx])

show = slice(0, 6 * 24 * 3)     # трое суток
t_axis = ts.iloc[te_idx].values[show]

fig, ax = plt.subplots(2, 1, figsize=(13, 7))
ax[0].plot(t_axis, y.iloc[te_idx].values[show], lw=1.3, color="black", label="Факт")
ax[0].plot(t_axis, pred[show], lw=1.1, color="#C44E52", label=f"Прогноз {best_name}")
ax[0].plot(t_axis, pred_naive[show], lw=0.9, color="#4C72B0", ls="--", alpha=0.8,
           label="Наивный прогноз")
ax[0].set_title(f"Факт и прогноз на тестовом горизонте (фолд 5), первые трое суток")
ax[0].set_xlabel("Время"); ax[0].set_ylabel("Вт·ч за 10 мин"); ax[0].legend(fontsize=8)

resid = y.iloc[te_idx].values - pred
ax[1].plot(ts.iloc[te_idx].values, resid, lw=0.5, color="#55A868")
ax[1].axhline(0, color="k", lw=1)
ax[1].set_title("Остатки (факт − прогноз) на всём тестовом фолде")
ax[1].set_xlabel("Время"); ax[1].set_ylabel("Остаток, Вт·ч")
plt.tight_layout(); plt.show()

print(f"Остатки: среднее = {resid.mean():+.2f} Вт·ч, std = {resid.std():.2f} Вт·ч")
print(f"Доля |остатка| > 100 Вт·ч: {(np.abs(resid) > 100).mean() * 100:.2f} %")
print(f"Максимальный промах: {np.abs(resid).max():.0f} Вт·ч "
      f"(в момент {ts.iloc[te_idx].values[np.abs(resid).argmax()]})")

**ВЫВОД 5 (заполните):**
_На каких участках прогноз точен, а где промахивается? Запаздывает ли модель за пиками (сдвиг прогноза вправо относительно факта)? Симметричны ли остатки или модель систематически занижает пики? Насколько велик выигрыш над наивным прогнозом и стоит ли он усложнения системы?_

## Блок 6. Важность признаков и энергосбережение — 2 балла

Ансамбли деревьев дают `feature_importances_` (снижение примеси при разбиениях). Но у `HistGradientBoostingRegressor` этого атрибута **нет** — для него применяется `permutation_importance`: признак перемешивается, и измеряется падение качества.

**Контрольный порог `rv1`.** Это столбец чистого шума, намеренно добавленный авторами датасета. Любой признак, важность которого не превышает важность `rv1`, следует считать неинформативным. Приём называется random control feature и является стандартом отбора признаков.

In [ ]:
# TODO-6: СТУДЕНТ — ВАЖНОСТЬ ПРИЗНАКОВ
#
# 1. Обучите модель бэггинга варианта на train последнего фолда (tr_idx).
# 2. Достаньте .named_steps['model'].feature_importances_, соберите pd.Series
#    с индексом X.columns, отсортируйте по убыванию -> переменная imp.
# 3. Постройте горизонтальный barh для топ-15 признаков;
#    красной пунктирной вертикалью отметьте важность контрольного шума rv1.
# 4. Выведите список признаков, НЕ прошедших порог rv1.
# 5. Сохраните важность rv1 в переменную noise_level.

imp = None
noise_level = None
# ВАШ КОД ЗДЕСЬ

In [ ]:
# ============ TEST 6 — важности признаков ============
def test_6_importance():
    assert imp is not None and noise_level is not None, "Заполните imp и noise_level"
    assert isinstance(imp, pd.Series) and len(imp) == X.shape[1], \
        "imp должна быть pd.Series по всем признакам"
    assert abs(imp.sum() - 1.0) < 1e-6, \
        "feature_importances_ ансамбля деревьев в сумме дают 1"
    assert imp.is_monotonic_decreasing, "Отсортируйте imp по убыванию"
    assert CONTROL_NOISE in imp.index, "Контрольный шум rv1 должен участвовать в сравнении"
    top = imp.index[0]
    assert imp[top] > imp[CONTROL_NOISE], "Топовый признак обязан превосходить шум"
    below = [c for c in imp.index if imp[c] <= noise_level and c != CONTROL_NOISE]
    print(f"✅ TEST 6 PASSED — топ признак: {top} ({imp[top]:.4f}), "
          f"порог шума rv1 = {noise_level:.4f}")
    print(f"   Признаков ниже порога шума: {len(below)} из {len(imp) - 1}")

test_6_importance()

## Блок 7. Выводы

Ответьте развёрнуто (**5–7 предложений на вопрос**), опираясь на **свои** числа.

---

**Вопрос 1. Насколько ML лучше наивного прогноза — и окупается ли это?**

Возьмите прирост MAE в Вт·ч, переведите его в кВт·ч за месяц (шаг 10 минут → 4320 интервалов в месяц) и в рубли по тарифу вашего региона. Сопоставьте с трудозатратами на разработку и сопровождение системы. Ответ «модель лучше на 12 %» без денег не засчитывается.

_ВАШ ОТВЕТ:_

---

**Вопрос 2. Как метеорологические параметры влияют на энергопотребление?**

Проанализируйте позиции `T_out` и других внешних признаков вашего варианта в рейтинге важности. Есть ли физическое объяснение? Как связаны наружная температура и нагрузка через отопление, вентиляцию и продолжительность светового дня? Не забудьте про инерцию: наружная температура влияет на потребление не мгновенно.

_ВАШ ОТВЕТ:_

---

**Вопрос 3. Что показывает контроль `rv1`?**

Сколько ваших признаков не прошли порог шума? Что это говорит о выбранном наборе зон? Стоит ли отказаться от части датчиков — и что это даст с точки зрения стоимости монтажа?

_ВАШ ОТВЕТ:_

---

**Вопрос 4. Стабильно ли качество по фолдам?**

Сравните метрики фолда 1 и фолда 5. Растёт качество с накоплением истории или деградирует? Что это значит для эксплуатации в течение учебного года — нужно ли переобучение и с какой периодичностью?

_ВАШ ОТВЕТ:_

---

**Вопрос 5. Три инженерных решения для завхоза школы.**

Сформулируйте три конкретных мероприятия по энергосбережению, вытекающих из важностей признаков и суточно-недельного профиля нагрузки. Для каждого укажите ожидаемый эффект и способ его измерения.

_ВАШ ОТВЕТ:_

In [ ]:
# 7.1. Финальная сводка.
print("=" * 74)
print(f"  ЛР №2 · Вариант {VARIANT} · источник данных: {DATA_SOURCE}")
print("=" * 74)
print(f"  Зоны              : {CFG['zone_set_id']} {CFG['sensors']}")
print(f"  Лаг / окно         : {CFG['lag']} / {CFG['window']} отсчётов "
      f"({CFG['window'] * 10} мин)")
print(f"  Модели             : {' | '.join(MODEL_NAMES)}")
print(f"  Горизонт           : {CFG['horizon'] * 10} минут")
print(f"  Признаков          : {X.shape[1]} | наблюдений: {len(X)}")
print(f"  Валидация          : TimeSeriesSplit(n_splits=5)")
print("=" * 74)
print(summary.to_string())
print("=" * 74)

folds_df.to_csv(f"lab02_folds_variant_{VARIANT}.csv", index=False, encoding="utf-8-sig")
summary.to_csv(f"lab02_summary_variant_{VARIANT}.csv", encoding="utf-8-sig")
print(f"Сохранено: lab02_folds_variant_{VARIANT}.csv, lab02_summary_variant_{VARIANT}.csv")

## Необязательное задание повышенной сложности

Повторите весь пайплайн для горизонта $h = 6$ (час вперёд), изменив `HORIZON = 6`, и ответьте:

1. Насколько деградировало качество всех моделей?
2. Насколько сильнее деградировал **наивный** прогноз по сравнению с обученными моделями?
3. Изменился ли состав топ-признаков? Выросла ли роль оконных статистик и календаря относительно лагов?
4. С какого горизонта наивный прогноз становится непригодным, а ML — необходимым?

Вывод оформите таблицей «горизонт × модель × RMSE».

## Чек-лист перед сдачей

- [ ] `VARIANT` соответствует моему номеру в ведомости
- [ ] Ряд отсортирован по времени, регулярность сетки проверена, разрывы обработаны через `ffill`
- [ ] Ни одного `shuffle=True`, `KFold` или `cross_val_score(cv=int)` в ноутбуке
- [ ] Ни одного `rolling(..., center=True)`
- [ ] `TEST 2` (детектор look-ahead bias) пройден
- [ ] Наивный baseline реализован и присутствует во всех таблицах
- [ ] Метрики приведены **по каждому фолду**, а не только усреднённо
- [ ] Построен график факт vs прогноз и график остатков
- [ ] Важности сопоставлены с контрольным шумом `rv1`
- [ ] Все шесть `TEST` выдают `✅ PASSED`
- [ ] Заполнены выводы 2.1, 2.2, 5 и пять вопросов блока 7
- [ ] `Runtime → Restart and run all` проходит без ошибок

**Сдача:** файл `lab_02_<Фамилия>_v<номер>.ipynb` в LMS курса.